In [ ]:
import nltk
import sklearn_crfsuite
from sklearn_crfsuite import metrics

# 1. Definizione delle Feature
# Un CRF ha bisogno di feature per ogni parola (es. la parola stessa, se è maiuscola, suffissi, ecc.)
# e informazioni sul contesto (parola precedente/successiva).
def word2features(sent, i):
    word = sent[i][0]
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],  # Suffix
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        word1 = sent[i-1][0]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
        })
    else:
        features['BOS'] = True  # Beginning of Sentence

    if i < len(sent)-1:
        word1 = sent[i+1][0]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
        })
    else:
        features['EOS'] = True  # End of Sentence

    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label in sent]

# 2. Caricamento Dati di Esempio (utilizzando NLTK Treebank)
nltk.download('treebank')
corpus = nltk.corpus.treebank.tagged_sents()

# Dividiamo in Train e Test set
train_sents = corpus[:3000]
test_sents = corpus[3000:]

X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

# 3. Addestramento del Modello CRF
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,  # Regolarizzazione L1
    c2=0.1,  # Regolarizzazione L2
    max_iterations=100,
    all_possible_transitions=True
)
crf.fit(X_train, y_train)

# 4. Valutazione
y_pred = crf.predict(X_test)
labels = list(crf.classes_)
labels.remove('-NONE-') # Rimuoviamo tag non informativi se presenti

sorted_labels = sorted(labels, key=lambda name: (name[1:], name[0]))
print(metrics.flat_classification_report(y_test, y_pred, labels=sorted_labels, digits=3))
